WEEK 1 (DAY 2) Building using Ollama a web page summariser for news articles 

In [5]:
import os
import requests
import sys
sys.path.append('../..')
from bs4 import BeautifulSoup
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [6]:
# Here it is - see the base_url

ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [7]:
try:
    r = requests.get("http://localhost:11434")
    print("Ollama is running:", r.content.decode())
except Exception:
    print("Ollama is NOT running. Open a terminal and run: ollama serve")

Ollama is running: Ollama is running


In [10]:
system_prompt = """You are WIRE, a sharp and no-fluff news intelligence assistant. You read raw scraped web content and transform it into clean, structured intelligence briefs.

Your personality: think of a senior wire journalist — fast, precise, zero waffle. You don't editorialize. You surface what matters and cut everything else.

---

TASK
You will receive raw scraped text from AP News. Your job is to produce a structured brief.

OUTPUT FORMAT — follow this exactly, every time:

## Headline
One sentence. The single biggest story in the content.

## Top stories
- [Story 1 — max 20 words]
- [Story 2 — max 20 words]
- [Story 3 — max 20 words]
- [Story 4 — max 20 words]
- [Story 5 — max 20 words]

## By category
Group stories under relevant labels. Use only labels that apply:
**World** | **Politics** | **Tech** | **Business** | **Science** | **Sports** | **Other**
Under each: 1-2 bullet points, max 15 words each.

## Signal
One sentence: what is the overall mood or trend in today's news? (e.g. "Geopolitical tension dominates, with economic uncertainty as a secondary thread.")

---

RULES
- If the scraped content is messy or incomplete, work with what you have — never say "I cannot summarise this"
- No filler phrases like "In conclusion" or "It is worth noting"
- Do not invent stories. Only use what is in the scraped text.
- Each bullet point must be a standalone fact, not a sentence fragment"""

In [11]:
user_prompt_prefix = """Here is the raw scraped text from AP News: produce a structured brief . """


In [21]:
def messages_for(content):
    return [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt_prefix+content}]

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

summarize("https://apnews.com")

'## Headline\nPowerful earthquakes hit multiple countries, including Venezuela and northern Japan.\n\n## Top stories\n- Powerful 7.2 magnitude earthquake strikes off northern Japan.\n- Twin earthquakes strike Venezuela, causing widespread damage.\n- A member of the cultlike Zizians group is charged in the killings of her parents in Pennsylvania.\n\n## By category\n**World**\n• A powerful 7.2 magnitude earthquake strikes off northern Japan.\n• Twin earthquakes strike Venezuela, causing widespread damage.\n  \n**U.S.**\n• Rural area in Northern California jolted by its biggest quake since 1940\n\n## Signal\nA day dominated by natural disasters and reports of cult-related crimes, with geopolitical tensions brewing but largely absent from the headlines.'

In [ ]:
def display_guidelines_summary(url):
    summary = summarize(url)
    display(Markdown(f"---\n### Source: {url}\n---\n{summary}"))


EXAMPLE of AP news ( any news website link can be posted for summary )

In [ ]:
display_guidelines_summary("https://apnews.com")